# 22 — Test CT-BERT xử lý URL và emoji
Dùng checkpoint seed 42 và calibrator đã khóa từ Notebook 21. Kiểm tra checksum, ID và nhãn của Test mirror trước khi suy luận. Ghi metric, dự đoán và checksum vào run mới. Không học lại tham số.

In [ ]:
from pathlib import Path
import sys, hashlib, time
root=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/ctbert_development.py').is_file())
sys.path.insert(0,str(root/'src'))
import ctbert_tweet_preprocess as ct

out=root/'results/20261124_ctbert_url_emoji_test_v1'
development=ct.OUT/'evaluation_lock.json'
base=ct.OUT/ct.KEY/'base_lock.json'
lock=ct.read(development)
assert lock['status']=='FROZEN_BEFORE_VALIDATION'
ct.verify(lock); ct.verify(ct.read(base)); ct.verify(ct.read(ct.OUT/'COMPLETE.json'))
methods=['ctbert_url_emoji']
revision='030deba36d118c124709fd72d7d5f14c7bc69053'
mirror=root/'data/raw/constraint2021_test_mirror'/revision
provenance=mirror/'provenance.json'
prov=ct.read(provenance)
assert prov['user_approved'] and prov['commit']==revision and prov['repo']=='diptamath/covid_fake_news'
historical=ct.unified.OUT/'evaluation/distilbert_Test2140_reused_predictions.csv'
prior_manifest=ct.read(ct.unified.OUT/'EVALUATION_COMPLETE.json')
if (out/'COMPLETE.json').is_file():
    ct.verify(ct.read(out/'COMPLETE.json'))
    display(ct.pd.read_csv(out/'metrics.csv'))
else:
    out.mkdir(parents=True,exist_ok=False)
    ct.save(out/'protocol.json',dict(methods=methods,development_lock_sha256=ct.old.sha(development),
        checkpoint_lock_sha256=ct.old.sha(base),mirror_commit=revision,provenance_sha256=ct.old.sha(provenance),
        independent_official_equivalence_verified=False,previously_used_test=True,tuning=False))
    tick=time.perf_counter()
    names=['Constraint_Test.csv','english_test_with_labels.csv']
    for name in names:
        info=next(row for row in prov['files'] if row['name']==name)
        b=(mirror/name).read_bytes()
        assert len(b)==info['bytes'] and hashlib.sha256(b).hexdigest()==info['sha256']
        assert hashlib.sha1(b'blob '+str(len(b)).encode()+bytes([0])+b).hexdigest()==info['git_blob_sha1']
    raw, test=(ct.pd.read_csv(mirror/name,dtype=str) for name in names)
    assert len(raw)==len(test)==2140 and raw.id.is_unique and test.id.is_unique
    assert set(raw.id)==set(test.id) and not test[['id','tweet','label']].isna().any().any()
    assert ct.np.array_equal(raw.set_index('id').loc[test.id,'tweet'],test.tweet)
    assert set(test.label)=={'fake','real'}
    test['sample_id']='test:'+test.id
    assert ct.old.sha(historical)==prior_manifest['files'][str(historical.relative_to(root))]
    ct.aligned(test,ct.pd.read_csv(historical))
    _,score,inference=ct.raw(ct.KEY,test.tweet.tolist())
    calibrator=ct.joblib.load(ct.OUT/'calibrator.joblib')
    assert list(calibrator.classes_)==['fake','real']
    p=calibrator.predict_proba(score)
    probabilities=[p]
    rows=[]
    for name,prob in zip(methods,probabilities):
        f=ct.frame(test,prob)
        m,bins=ct.pm.evaluate(f)
        rows.append(dict(model=name,split='Test2140_reused_mirror',**m))
        f.to_csv(out/f'{name}_predictions.csv',index=False,mode='x')
        bins.to_csv(out/f'{name}_calibration_bins.csv',index=False,mode='x')
    result=ct.pd.DataFrame(rows)
    result.to_csv(out/'metrics.csv',index=False,mode='x')
    ct.save(out/'COMPLETE.json',dict(status='COMPLETE',n=len(test),label_counts=test.label.value_counts().to_dict(),
        ctbert_inference_seconds=inference,total_seconds=time.perf_counter()-tick,
        distilbert_prediction_sha256=ct.old.sha(historical),previously_used_test=True,
        files={str(p.relative_to(root)):ct.old.sha(p) for p in out.iterdir() if p.is_file()}))
    display(result[['model','weighted_f1','macro_f1','accuracy','mcc','log_loss','brier_real_binary']])
